In [ ]:
import pandas as pd
import json
import time
import os
from datetime import datetime
from dashscope import Generation

# ====================== 配置 ======================
API_KEY = "sk-6ba242c2cf004c37af7ac0ae895b9844"   # ←←← 改成你的真实 Key

MODEL_NAME = "qwen-turbo"
TEMPERATURE = 0.25
SEED = 42
MAX_TOKENS = 1000

CHUNK_SIZE = 200                  # 建议改成200，平衡速度与安全性
INPUT_FILE = r"D:\SEMB\5508\cleaned2025.csv"
OUTPUT_FILE = "analyzed_cleaned.csv"   # 改成 .csv

# ====================== 完整 Prompt（已包含你所有定义） ======================
SYSTEM_PROMPT = """你是一个专业的网络评论（尤其是LIHKG）情感与指责分析助手。

请严格按照以下 JSON 格式输出，不要添加任何额外文字：

{
  "sentiment_category": "positive / negative / neutral",
  "fine_grained_emotions": ["anger", "sad", "joy", "surprise", "fear", "mock"],
  "sentiment_score": -0.85,
  "has_blame": "yes / no",
  "blame_subject": "网民 / 受害者/家属/亲历者 / 政府/警方/平台者 / 同情诈骗者/另类立场",
  "blame_object": "诈骗集团/骗徒 / 受害者本人 / 警方/政府/监管部门 / 电信/平台/银行 / 境外势力/东南亚园区 / 无明确对象",
  "blame_intensity": "weak / medium / strong"
}

**sentiment_score 规则**（必须严格遵守）：
- 极度负面（强烈愤怒、嘲讽）：-0.8 ~ -1.0
- 明显负面（不满、批评）：-0.4 ~ -0.8
- 中性或轻微负面：-0.2 ~ 0.2
- 轻微正面：0.2 ~ 0.5
- 明显正面：0.5 ~ 0.8

**blame_subject 定义**：
1. 网民：纯路人、普通网民，愤怒、吐槽、同情受害者。
2. 受害者/家属/亲历者：自称或暗示自己/家人被骗，带个人经历细节。
3. 政府/警方/平台者：指责监管不力、执法慢。
4. 同情诈骗者/另类立场：同情骗子、生活所迫、洗白诈骗者。

**blame_object 定义**：
1. 诈骗集团/骗徒：直接骂骗子、诈骗团伙。
2. 受害者本人：怪受害者贪心、蠢。
3. 警方/政府/监管部门：监管不力。
4. 电信/平台/银行：泄露信息。
5. 境外势力/东南亚园区：缅北、柬埔寨等。
6. 无明确对象：泛骂社会、道德沦丧。

只输出 JSON 对象。"""

# ====================== 调用 API ======================
def analyze_comment(content: str) -> dict:
    if not content or pd.isna(content):
        return {
            "sentiment_category": "neutral",
            "fine_grained_emotions": [],
            "sentiment_score": 0.0,
            "has_blame": "no",
            "blame_subject": "网民",
            "blame_object": "无明确对象",
            "blame_intensity": "weak"
        }

    prompt = f"请分析以下网络评论：\n\n{content}"

    try:
        response = Generation.call(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": SYSTEM_PROMPT},
                      {"role": "user", "content": prompt}],
            api_key=API_KEY,
            temperature=TEMPERATURE,
            seed=SEED,
            max_tokens=MAX_TOKENS,
            result_format="message"
        )

        raw_text = response.output.choices[0].message.content.strip()

        if raw_text.startswith("```"):
            raw_text = raw_text.split("```")[1].strip() if len(raw_text.split("```")) > 1 else raw_text

        result = json.loads(raw_text)
        return result
    except Exception as e:
        print(f"API调用失败: {e}")
        return {
            "sentiment_category": "neutral",
            "fine_grained_emotions": [],
            "sentiment_score": 0.0,
            "has_blame": "no",
            "blame_subject": "网民",
            "blame_object": "无明确对象",
            "blame_intensity": "weak"
        }

# ====================== 主程序（.csv 版本 + 断点续跑） ======================
def main():
    if not os.path.exists(INPUT_FILE):
        print(f"错误：输入文件 {INPUT_FILE} 不存在！")
        return

    df_input = pd.read_csv(INPUT_FILE)
    print(f"输入文件共有 {len(df_input)} 行数据")

    # 加载已分析结果（支持断点续跑）
    if os.path.exists(OUTPUT_FILE):
        df_output = pd.read_csv(OUTPUT_FILE)
        processed_ids = set(df_output["comment_id"].astype(str))
        print(f"已处理 {len(processed_ids)} 条评论，继续处理剩余数据")
    else:
        df_output = pd.DataFrame()
        processed_ids = set()
        print("首次运行，创建新分析文件")

    # 找出未处理的行
    df_input["comment_id"] = df_input["comment_id"].astype(str)
    to_process = df_input[~df_input["comment_id"].isin(processed_ids)]

    if len(to_process) == 0:
        print("所有数据已处理完毕！")
        return

    print(f"本次需处理 {len(to_process)} 条新评论\n")

    for start_idx in range(0, len(to_process), CHUNK_SIZE):
        chunk = to_process.iloc[start_idx : start_idx + CHUNK_SIZE]
        new_rows = []

        for _, row in chunk.iterrows():
            comment_id = str(row["comment_id"])

            content = None
            for col in ["content_x", "translated_content_x", "content_y", "content"]:
                if col in row and pd.notna(row[col]) and str(row[col]).strip():
                    content = str(row[col]).strip()
                    break

            if not content:
                continue

            print(f"分析中 → {comment_id}")
            analysis = analyze_comment(content)

            analyzed_row = row.to_dict()
            analyzed_row.update({
                "sentiment_category": analysis["sentiment_category"],
                "fine_grained_emotions": ",".join(analysis.get("fine_grained_emotions", [])),
                "sentiment_score": round(analysis["sentiment_score"], 2),
                "has_blame": analysis["has_blame"],
                "blame_subject": analysis["blame_subject"],
                "blame_object": analysis["blame_object"],
                "blame_intensity": analysis["blame_intensity"],
                "analysis_timestamp": datetime.now().isoformat()
            })
            new_rows.append(analyzed_row)

            time.sleep(0.5)   # 轻微延时，避免限流

        # 保存当前 chunk（追加模式，更安全）
        if new_rows:
            new_df = pd.DataFrame(new_rows)
            if not df_output.empty:
                df_output = pd.concat([df_output, new_df], ignore_index=True)
            else:
                df_output = new_df

            # 去重 + 排序后保存
            df_output = df_output.drop_duplicates(subset=["comment_id"]).sort_values(by="comment_id")
            df_output.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
            print(f"✅ 已保存进度 → 共 {len(df_output)} 条记录\n")

    print("🎉 全部处理完成！输出文件：", OUTPUT_FILE)

if __name__ == "__main__":
    print("=== 通义千问评论分析脚本启动（CSV安全版本） ===")
    main()